# പാഠം 09 - മെറ്റാകോഗ്നിഷൻ ഡിസൈൻ മാതൃകം


## ക്രമീകരണം

ഈ നോട്ട്ബുക്ക് Microsoft Agent Framework ഉപയോഗിച്ച് Metacognition രൂപകൽപ്പന മാതൃക പ്രദർശിപ്പിക്കുന്നു.

**മുൻനിബന്ധനകൾ:**
- Azure OpenAI ഡിപ്ലോയ്മെന്റ് environment variables വഴി കോൺഫിഗർ ചെയ്തിരിക്കുന്നത്
- Azure CLI ലോഗിൻ ചെയ്തിരിക്കണം (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## മെറ്റാകോഗ്നിഷൻ എന്താണ്?

മെറ്റാകോഗ്നിഷൻ എന്നത് **ചിന്തയെക്കുറിച്ച് ചിന്തിക്കുക**. AI ഏജന്റുകളുടെ സന്ദർഭത്തിൽ, ഇത് അർത്ഥമാക്കുന്നത് താഴെപ്പറയുന്ന കാര്യങ്ങൾ ചെയ്യാൻ കഴിയുന്ന ഏജന്റുകൾ നിർമ്മിക്കുക എന്നതാണ്:

- **സ്വയം പ്രതിഫലനം ചെയ്യുക** അവരുടെ സ്വന്തം ഫലങ്ങളിലും ചിന്താ പ്രക്രിയയിലും
- **തെറ്റുകൾ കണ്ടെത്തുക** മൗനമായി പരാജയപ്പെടുന്നതിന് പകരം സൗമ്യമായി പുനരുദ്ധരിക്കുക
- **വിലയിരുത്തുക** അവരുടെ പ്രതികരണങ്ങൾ പൂര്‍ണവും സഹായകരവുമാണോ എന്നത്
- **സമയനുസരിച്ച് അനുകുലീകരിക്കുക** അവരുടെ തന്ത്രം ഒരു പ്രാഥമിക സമീപനം പ്രവർത്തിക്കാതെ പോകുമ്പോൾ (ഉദാ., ബാക്കപ്പ് സിസ്റ്റത്തിലേക്ക് തിരിച്ചുപോകൽ)

ഒരു മെറ്റാകോഗ്നിറ്റീവ് ഏജന്റ് വെറും ചോദ്യങ്ങൾക്ക് മറുപടി പറയുന്നവനല്ല — അത് തന്റെ തന്നെ പ്രകടനം നിരീക്ഷിക്കുകയും ഉടൻ ക്രമീകരിക്കുകയും ചെയ്യുന്നു.


## പ്രാഥമികവും ബാക്കപ് ഉപകരണങ്ങളും

ഒരു സാധാരണ മെറ്റാകോഗ്നിഷൻ മാതൃകയാണ് **ഫാൾബാക്ക് തന്ത്രം**. ഏജന്റ് ആദ്യം ഒരു പ്രാഥമിക ഉപകരണമായി ശ്രമിക്കും; അത് പരാജയപ്പെടുകയാണെങ്കിൽ (ഉദാ., ഒരു 404 പിശക്), ഏജന്റ് പരാജയം തിരിച്ചറിയുകയും പാരദർശകമായി ഒരു ബാക്കപ് ഉപകരണത്തിലേക്ക് മാറുകയും ചെയ്യും.

ഇത് യാഥാർത്ഥ്യത്തിലെ സിസ്റ്റങ്ങൾ പ്രതിഫലിപ്പിക്കുന്നു, അവിടെ പ്രാഥമിക സേവനങ്ങൾ ലഭ്യമല്ലായിരിക്കാം, അതിനാൽ ഏജന്റുകൾക്ക് പ്രശ്നം സ്വയം നിർണയിച്ച് ഒരു ആശ്രയവഴി തിരഞ്ഞെടുത്ത് മാറേണ്ടി വരും.

താഴെ ഞങ്ങൾ വിമാനാന്വേഷണത്തിന് ഉപയോഗിക്കുന്ന രണ്ട് ഉപകരണങ്ങൾ നിർവചിക്കുന്നു:
- **പ്രാഥമികം** — പാരീസ്, ടോക്കിയോ, ബാർസലോന എന്നിവ ഉൾക്കൊള്ളുന്നു
- **ബാക്കപ്** — ബർലിൻ, സിഡ്നി, ന്യൂയോർക്ക് സിറ്റി എന്നിവ ഉൾക്കൊള്ളുന്നു


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## തെറ്റ് വീണ്ടെടുക്കൽ ഉൾക്കൊള്ളുന്ന സ്വയംപരിശോധന ഏജന്റ്

താഴെ കാണുന്ന ഏജന്റിനോട് ആദ്യം പ്രാഥമിക ഫ്ലൈറ്റ് സിസ്റ്റം പരീക്ഷിക്കാൻ, പരാജയങ്ങൾ തിരിച്ചറിയാൻ, സുതാര്യമായി ബാക്കപ്പ് സിസ്റ്റത്തിലേക്ക് മടങ്ങാൻ നിർദ്ദേശിച്ചിട്ടുണ്ട്. ഓരോ മറുപടിക്കും ശേഷം അത് സംക്ഷിപ്തമായി സ്വയം മൂല്യനിർണയം നടത്തുന്നു — ഉപയോക്താവിന്റെ ചോദ്യത്തിന് ഇത് പൂർണമായി മറുപടി നൽകിയോയെന്ന്.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## സ്വയം-അവലോകന മാതൃകം

മേറ്റാകോഗ്നിഷന്റെ മറ്റൊരു വശം **സ്വയം-അവലോകനമാണ്**: വേറിട്ടൊരു ഏജന്റ് (അഥവാ രണ്ടാമത്തെ പാസ്‌യിൽ അതേ ഏജന്റ്) ഒരു പ്രതികരണം പൂർണതയ്ക്കും കൃത്യതയ്ക്കും സഹായപ്രദതയ്ക്കും വിലയിരുത്തുന്നു.

താഴെ നാം `ResponseEvaluator` ഏജന്റിനെ സൃഷ്ടിക്കുന്നു, അത് യാത്രാ ഏജന്റിന്റെ പ്രതികരണങ്ങളെ മൂന്ന് ആയാമങ്ങളിൽ സ്‌കോർ ചെയ്യുന്നു.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework ഉപയോഗിച്ച് **മെറ്റക്കോഗ്നിറ്റീവ് ഏജന്റുകൾ** എങ്ങനെ നിർമ്മിക്കാമെന്നു പഠിച്ചു:

- **സ്വയം-പരിശോധന**: തങ്ങളുടെ സ്വന്തം ചിന്തനത്തെ നിരീക്ഷിച്ച് സംഭവിച്ചതെല്ലാം വ്യക്തമായി അറിയിക്കുന്ന ഏജന്റുകൾ.
- **ഫോൾബാക്കുകളോടുകൂടിയ പിശക് പുനരുദ്ധാരണം**: പ്രാഥമികവും ബാക്കപ്പും ഉള്ള ടൂൾ മാതൃകയിൽ ഏജന്റ് പരാജയങ്ങൾ (ഉദാ., 404 errors) തിരിച്ചറിഞ്ഞ് സ്വയം മറ്റൊരു ഉറവിടം പരീക്ഷിക്കുന്നു.
- **സ്വയം-മൂല്യനിർണ്ണയം**: ഒരു വേർതിരിച്ച മൂല്യനിർണ്ണയ ഏജന്റ്, പ്രതികരണങ്ങൾക്ക് സമ്പൂർണത, കൃത്യത, സഹായപ്രദത എന്നിവയുടെ അടിസ്ഥാനത്തിൽ സ്‌കോർ നൽകുന്നു.

ഈ മാതൃകകൾ ഏജന്റുകളെ കൂടുതൽ ദൃഢതയുള്ളതും പാരദർശകവുമായതും വിശ്വസനീയവുമായതും ആക്കുന്നു — ഉത്പാദന വിന്യാസങ്ങൾക്ക് നിർണായകമായ ഗുണങ്ങൾ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ഡിസ്‌ക്ലെയിമർ:
ഈ രേഖ AI പരിഭാഷാ സേവനം Co-op Translator (https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷ ചെയ്തതാണ്. ഞങ്ങൾ കൃത്യതക്ക് ശ്രമിച്ചിരുന്നെങ്കിലും, കമ്മിറ്റി പരിഭാഷകളില്‍ പിശകുകൾ അല്ലെങ്കിൽ അസംബന്ധതകൾ ഉണ്ടായിരിക്കാമെന്നത് ദയവായി മനസ്സിലാക്കുക. യഥാർത്ഥ ഭാഷയിൽ ഉള്ള പകർപ്പ് ആണ് അധികാരം ഉള്ള സ്രോതസ്സായി കണക്കാക്കേണ്ടത്. നിർണായക വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ചതിന്റെ ഫലമായി ഉണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണങ്ങൾക്കോ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കോ ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
